# LeetCode 318 — Maximum Product of Word Lengths bằng Python

Link đề: https://leetcode.com/problems/maximum-product-of-word-lengths/ · Độ khó: **Medium** · Chủ đề: Array, String, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def maxProduct(self, words: List[str]) -> int:
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong mảng `words` chọn **hai từ không có chữ cái chung** sao cho **tích độ dài** `len(a) · len(b)` lớn nhất; nếu không có cặp nào, trả `0`.
2. Biết kỹ thuật **bitmask cho tập chữ cái**: mỗi từ → một số nguyên 26 bit; hai từ **không có chữ chung** ⇔ `mask_a & mask_b == 0`.
3. Đi từ **kiểm tra chung chữ bằng `set` cho mọi cặp** → **bitmask AND cho mọi cặp** → **gộp các từ cùng mặt nạ (chỉ giữ độ dài lớn nhất)** → **sắp theo độ dài giảm dần + cắt nhánh** → **bitset theo từ cho mỗi chữ cái (tra bù trực tiếp)**.
4. Hiểu **vì sao chỉ giữ độ dài lớn nhất cho mỗi mặt nạ** là đủ, và vì sao cắt nhánh theo độ dài là đúng.
5. Tránh lỗi kinh điển: dùng số chữ cái **khác nhau** thay vì độ dài, dùng `+=` thay `|=` khi dựng mặt nạ, đảo điều kiện `& == 0`, và ghi đè mặt nạ trùng bằng độ dài ngắn hơn.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words`, trả về **giá trị lớn nhất của `length(word[i]) * length(word[j])`** sao cho hai từ `word[i]` và `word[j]` **không có chữ cái chung**. Nếu không có hai từ như vậy, trả về `0`.

**Ví dụ 1:** `words = ["abcw","baz","foo","bar","xtfn","abcdef"]` → `16`. Hai từ `"abcw"` và `"xtfn"`.

**Ví dụ 2:** `words = ["a","ab","abc","d","cd","bcd","abcd"]` → `4`. Hai từ `"ab"` và `"cd"`.

**Ví dụ 3:** `words = ["a","aa","aaa","aaaa"]` → `0` (mọi cặp đều có chữ `a`).

**Ràng buộc:** `2 <= words.length <= 1000`; `1 <= words[i].length <= 1000`; `words[i]` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
import random

examples = [
    (["abcw", "baz", "foo", "bar", "xtfn", "abcdef"], 16),
    (["a", "ab", "abc", "d", "cd", "bcd", "abcd"], 4),
    (["a", "aa", "aaa", "aaaa"], 0),
]
for words, e in examples:
    print("words =", words, "-> kỳ vọng", e)

words = ['abcw', 'baz', 'foo', 'bar', 'xtfn', 'abcdef'] -> kỳ vọng 16
words = ['a', 'ab', 'abc', 'd', 'cd', 'bcd', 'abcd'] -> kỳ vọng 4
words = ['a', 'aa', 'aaa', 'aaaa'] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Tập chữ cái của một từ

Hai từ "không có chữ chung" nghĩa là **hai tập chữ cái rời nhau**. Độ dài của từ (`len`) tính cả chữ lặp, còn tập chữ cái chỉ quan tâm chữ nào xuất hiện. Ví dụ `"aaaa"` có độ dài `4` nhưng tập chữ cái chỉ là `{a}`.

In [2]:
for w in ("abcw", "aaaa", "bar"):
    print(repr(w), "độ dài", len(w), "| tập chữ cái", sorted(set(w)))
print("rời nhau?", set("abcw").isdisjoint("xtfn"), set("abcw").isdisjoint("baz"))

'abcw' độ dài 4 | tập chữ cái ['a', 'b', 'c', 'w']
'aaaa' độ dài 4 | tập chữ cái ['a']
'bar' độ dài 3 | tập chữ cái ['a', 'b', 'r']
rời nhau? True False


### 3.2. Mặt nạ bit (bitmask) 26 bit

Chữ `'a'` ứng với bit 0, `'b'` bit 1, ..., `'z'` bit 25. Mặt nạ của một từ là OR của các bit của từng chữ: `mask |= 1 << (ord(c) - ord('a'))`. Khi đó: `mask_a & mask_b == 0` ⇔ hai từ không có chữ chung. Phép AND trên số nguyên chỉ `O(1)`.

In [3]:
def word_mask(w):
    m = 0
    for c in w:
        m |= 1 << (ord(c) - 97)
    return m


for w in ("a", "b", "ab", "aab", "z", "abcw"):
    print(f"{w!r:7} mask = {word_mask(w):>8} = {word_mask(w):026b}")
print("ab & cd =", word_mask("ab") & word_mask("cd"), "| ab & bc =", word_mask("ab") & word_mask("bc"))

'a'     mask =        1 = 00000000000000000000000001
'b'     mask =        2 = 00000000000000000000000010
'ab'    mask =        3 = 00000000000000000000000011
'aab'   mask =        3 = 00000000000000000000000011
'z'     mask = 33554432 = 10000000000000000000000000
'abcw'  mask =  4194311 = 00010000000000000000000111
ab & cd = 0 | ab & bc = 2


### 3.3. Tính chất của `|=`: lặp chữ không đổi mặt nạ

OR là **idempotent** (`x | x = x`): chữ lặp lại không làm đổi mặt nạ, nên `"aab"` và `"ab"` có **cùng mặt nạ**. Dùng `+=` thay cho `|=` sẽ làm hỏng điều này (lỗi hay gặp).

In [4]:
print(word_mask("aab") == word_mask("ab") == word_mask("abba"))

True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 2: `["a","ab","abc","d","cd","bcd","abcd"]`.

| từ | tập chữ cái | độ dài |
|---|---|---|
| `a` | `{a}` | 1 |
| `ab` | `{a,b}` | 2 |
| `abc` | `{a,b,c}` | 3 |
| `d` | `{d}` | 1 |
| `cd` | `{c,d}` | 2 |
| `bcd` | `{b,c,d}` | 3 |
| `abcd` | `{a,b,c,d}` | 4 |

Các cặp rời nhau: `a–d` (tích 1), `a–cd` (2), `a–bcd` (3), `ab–d` (2), `ab–cd` (4), `abc–d` (3). Lớn nhất `ab–cd = 4`. (`abcd` không rời với từ nào vì chứa cả 4 chữ.)

Để so hai từ nhanh: đổi mỗi từ thành **một số nguyên**, hai từ rời nhau khi `AND` bằng 0.

In [5]:
words = ["a", "ab", "abc", "d", "cd", "bcd", "abcd"]
best = 0
for i in range(len(words)):
    for j in range(i + 1, len(words)):
        if word_mask(words[i]) & word_mask(words[j]) == 0:
            print(f"{words[i]!r:7} - {words[j]!r:7} rời nhau, tích = {len(words[i]) * len(words[j])}")
            best = max(best, len(words[i]) * len(words[j]))
print("lớn nhất:", best)

'a'     - 'd'     rời nhau, tích = 1
'a'     - 'cd'    rời nhau, tích = 2
'a'     - 'bcd'   rời nhau, tích = 3
'ab'    - 'd'     rời nhau, tích = 2
'ab'    - 'cd'    rời nhau, tích = 4
'abc'   - 'd'     rời nhau, tích = 3
lớn nhất: 4


## 5. Cách 1 — Brute Force / baseline: kiểm tra từng cặp bằng `set`

**Ý tưởng.** Với mọi cặp `i < j`: dùng `set(words[i]).isdisjoint(words[j])` (hoặc phép giao) để kiểm tra có chữ chung không; nếu rời nhau thì cập nhật tích lớn nhất. Dựng `set` mỗi lần là lãng phí nên dựng trước.

In [6]:
def max_product_brute(words: List[str]) -> int:
    sets = [set(w) for w in words]
    best = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if sets[i].isdisjoint(sets[j]):
                best = max(best, len(words[i]) * len(words[j]))
    return best


for words, e in examples:
    assert max_product_brute(words) == e
print("PASS ví dụ (brute force)")

PASS ví dụ (brute force)


**Độ phức tạp (Cách 1).** Dựng tập `O(L)` (`L` tổng độ dài); kiểm tra `n²/2` cặp, mỗi cặp `O(26)` ⇒ `O(L + 26·n²)`; Space `O(26·n)`. Với `n = 1000` là `~5·10⁵` cặp: vẫn chạy được, nhưng chậm hơn bitmask nhiều.

**Nút thắt.** Phép "rời nhau" của hai tập tốn `O(26)`. Bằng mặt nạ bit có thể kiểm tra trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Mặt nạ bit:** mỗi từ → số nguyên `mask`; "rời nhau" ⇔ `mask_a & mask_b == 0` trong `O(1)`.
2. **Gộp theo mặt nạ:** hai từ có **cùng mặt nạ** là hoán đổi được cho nhau về mặt "chữ chung"; với một mặt nạ chỉ cần giữ **từ dài nhất**. Số mặt nạ khác nhau thường ít hơn `n`.
3. **Cắt nhánh:** sắp từ theo độ dài giảm dần; khi `len_i · len_j ≤ best` (với `j` đang đi xuống độ dài giảm) thì mọi `j` tiếp theo còn tệ hơn: dừng sớm.
4. **Bitset theo chữ cái:** với mỗi chữ `c` giữ một số nguyên lớn `bits[c]` có bit `r` bật nếu từ hạng `r` (theo độ dài giảm dần) chứa `c`. Từ `i` tương thích với những từ **không** nằm trong OR của `bits[c]` với `c ∈ i`; từ **dài nhất** trong số đó là bit thấp nhất.

In [7]:
# Minh họa: số mặt nạ khác nhau thường ít hơn số từ
rnd = random.Random(318)
words = ["".join(rnd.choice("abcdef") for _ in range(rnd.randint(1, 8))) for _ in range(1000)]
print("số từ:", len(words), "| số mặt nạ khác nhau:", len({word_mask(w) for w in words}), "(tối đa 2^6 - 1 = 63 vì chỉ có 6 chữ)")

số từ: 1000 | số mặt nạ khác nhau: 63 (tối đa 2^6 - 1 = 63 vì chỉ có 6 chữ)


## 7. Các cách tiếp theo

### Cách 2 — Bitmask AND cho mọi cặp (cách của lời giải tham khảo)

**Giải thích.** Tính `mask[i]` cho từng từ rồi duyệt mọi cặp `i < j`: nếu `mask[i] & mask[j] == 0` thì cập nhật `best = max(best, len_i · len_j)`.

In [8]:
def max_product_masks(words: List[str]) -> int:
    n = len(words)
    masks = []
    for w in words:
        m = 0
        for c in w:
            m |= 1 << (ord(c) - 97)
        masks.append(m)
    lens = [len(w) for w in words]
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            if masks[i] & masks[j] == 0:
                best = max(best, lens[i] * lens[j])
    return best


for words, e in examples:
    assert max_product_masks(words) == e
print("PASS ví dụ (bitmask cặp)")

PASS ví dụ (bitmask cặp)


**Độ phức tạp Cách 2:** Dựng mặt nạ `O(L)`; duyệt cặp `O(n²)`; Space `O(n)`.

### Cách 3 — Gộp các từ cùng mặt nạ (chỉ giữ độ dài lớn nhất)

**Giải thích.** Dùng `dict`: `best_len[mask] = max(len)`. Sau đó duyệt mọi cặp **mặt nạ khác nhau** (hai từ cùng mặt nạ không thể rời nhau vì mặt nạ khác 0, nên không cần xét cặp cùng mặt nạ).

In [9]:
def max_product_dedupe(words: List[str]) -> int:
    best_len = {}
    for w in words:
        m = 0
        for c in w:
            m |= 1 << (ord(c) - 97)
        if len(w) > best_len.get(m, 0):
            best_len[m] = len(w)
    items = list(best_len.items())
    best = 0
    for i in range(len(items)):
        mi, li = items[i]
        for j in range(i + 1, len(items)):
            mj, lj = items[j]
            if mi & mj == 0:
                best = max(best, li * lj)
    return best


for words, e in examples:
    assert max_product_dedupe(words) == e
print("PASS ví dụ (gộp mặt nạ)")

PASS ví dụ (gộp mặt nạ)


**Độ phức tạp Cách 3:** Time `O(L + d²)` với `d ≤ n` là số mặt nạ khác nhau; Space `O(d)`.

### Cách 4 — Sắp theo độ dài giảm dần + cắt nhánh

**Giải thích.** Sắp `(độ dài, mặt nạ)` giảm dần theo độ dài. Với `i`, duyệt `j > i`: nếu `len_i · len_j ≤ best` thì `break` (các `j` sau còn ngắn hơn). Nếu `mask_i & mask_j == 0` thì cập nhật `best` rồi `break` (các `j` sau cho tích nhỏ hơn). Ngoài ra nếu `len_i · len_{i+1} ≤ best` có thể dừng hẳn vòng ngoài (các `i` sau cũng ngắn hơn).

In [10]:
def max_product_pruned(words: List[str]) -> int:
    items = []
    for w in words:
        m = 0
        for c in w:
            m |= 1 << (ord(c) - 97)
        items.append((len(w), m))
    items.sort(reverse=True)                              # độ dài giảm dần
    best = 0
    n = len(items)
    for i in range(n):
        li, mi = items[i]
        if i + 1 < n and li * items[i + 1][0] <= best:
            break                                          # mọi cặp từ i trở đi tích nhỏ hơn hoặc bằng
        for j in range(i + 1, n):
            lj, mj = items[j]
            if li * lj <= best:
                break
            if mi & mj == 0:
                best = li * lj
                break
    return best


for words, e in examples:
    assert max_product_pruned(words) == e
print("PASS ví dụ (sắp + cắt nhánh)")

PASS ví dụ (sắp + cắt nhánh)


**Độ phức tạp Cách 4:** Sắp xếp `O(n log n)`; cặp duyệt thực tế rất ít nhờ cắt nhánh, xấu nhất `O(n²)` (khi hầu như không có cặp rời nhau mà tích vẫn lớn hơn `best = 0`); Space `O(n)`.

### Cách 5 — Bitset theo chữ cái (tra "bù" trực tiếp)

**Giải thích.** Sắp từ theo độ dài giảm dần, hạng `r = 0..n−1`. Với mỗi chữ cái `c` giữ số nguyên `bits[c]`: bit `r` bật nếu từ hạng `r` chứa `c`. Với từ hạng `r` có tập chữ `S`: `conflict = OR của bits[c] với c ∈ S`; `free = ((1 << n) − 1) & ~conflict`. Từ tương thích **dài nhất** là bit **thấp nhất** của `free` (hạng nhỏ nhất ⇒ dài nhất). Mỗi từ `O(26)` phép trên số nguyên lớn `n` bit.

In [11]:
def max_product_bitsets(words: List[str]) -> int:
    order = sorted(range(len(words)), key=lambda i: -len(words[i]))
    ranked = [words[i] for i in order]
    n = len(ranked)
    bits = [0] * 26
    for r, w in enumerate(ranked):
        for c in set(w):
            bits[ord(c) - 97] |= 1 << r
    full = (1 << n) - 1
    best = 0
    for r, w in enumerate(ranked):
        conflict = 0
        for c in set(w):
            conflict |= bits[ord(c) - 97]
        free = full & ~conflict
        if free:
            partner = (free & -free).bit_length() - 1      # bit thấp nhất = từ dài nhất tương thích
            best = max(best, len(w) * len(ranked[partner]))
    return best


for words, e in examples:
    assert max_product_bitsets(words) == e
print("PASS ví dụ (bitset theo chữ cái)")

PASS ví dụ (bitset theo chữ cái)


**Độ phức tạp Cách 5:** Time `O(L + n · 26 · n / 64)` (mỗi phép OR/AND số nguyên `n` bit tốn `O(n/64)`); Space `O(26 · n / 64 + n)`. Không có vòng lặp cặp `O(n²)` tường minh.

> Cách 2 là cách nộp bài (ngắn, đúng). Cách 3 và 4 tối ưu thêm. Cách 5 là kỹ thuật bitset mạnh khi `n` lớn. Cách 1 làm oracle.

## 8. TRACE từng bước

Theo dõi Cách 2 trên `words = ["a","ab","abc","d","cd","bcd","abcd"]`: in mặt nạ nhị phân và từng cặp rời nhau.

In [12]:
def trace_pairs(words):
    masks = [word_mask(w) for w in words]
    for w, m in zip(words, masks):
        print(f"{w!r:7} len={len(w)} mask={m:04b}")
    best = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if masks[i] & masks[j] == 0:
                prod = len(words[i]) * len(words[j])
                best = max(best, prod)
                print(f"  {words[i]!r}-{words[j]!r}: {masks[i]:04b} & {masks[j]:04b} = 0 -> tích {prod}, best = {best}")
    return best


assert trace_pairs(["a", "ab", "abc", "d", "cd", "bcd", "abcd"]) == 4

'a'     len=1 mask=0001
'ab'    len=2 mask=0011
'abc'   len=3 mask=0111
'd'     len=1 mask=1000
'cd'    len=2 mask=1100
'bcd'   len=3 mask=1110
'abcd'  len=4 mask=1111
  'a'-'d': 0001 & 1000 = 0 -> tích 1, best = 1
  'a'-'cd': 0001 & 1100 = 0 -> tích 2, best = 2
  'a'-'bcd': 0001 & 1110 = 0 -> tích 3, best = 3
  'ab'-'d': 0011 & 1000 = 0 -> tích 2, best = 3
  'ab'-'cd': 0011 & 1100 = 0 -> tích 4, best = 4
  'abc'-'d': 0111 & 1000 = 0 -> tích 3, best = 4


Mặt nạ (4 bit `dcba`): `a = 0001`, `ab = 0011`, `abc = 0111`, `d = 1000`, `cd = 1100`, `bcd = 1110`, `abcd = 1111`. Ví dụ `ab & cd = 0011 & 1100 = 0000` nên rời nhau (tích `2 · 2 = 4`); `abc & bcd = 0110 ≠ 0` nên có chữ chung.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (bitmask cho mọi cặp), đúng chữ ký của LeetCode.

In [13]:
class Solution:
    def maxProduct(self, words: List[str]) -> int:
        n = len(words)
        masks = [0] * n
        lens = [len(w) for w in words]
        for i, w in enumerate(words):
            for c in w:
                masks[i] |= 1 << (ord(c) - ord("a"))
        best = 0
        for i in range(n):
            for j in range(i + 1, n):
                if masks[i] & masks[j] == 0:
                    best = max(best, lens[i] * lens[j])
        return best


for words, e in examples:
    assert Solution().maxProduct(words) == e
print("PASS ví dụ đề cho")

PASS ví dụ đề cho


## 10. Vì sao thuật toán đúng

- **Mặt nạ đại diện đúng cho tập chữ cái:** bit `k` bật ⇔ chữ thứ `k` xuất hiện trong từ. `mask_a & mask_b` có bit `k` ⇔ cả hai từ chứa chữ thứ `k`. Vậy `& == 0` ⇔ không có chữ chung.
- **Duyệt mọi cặp:** Cách 2 xét tất cả `i < j` nên bao gồm cặp tối ưu.
- **Gộp theo mặt nạ (Cách 3):** hai từ cùng mặt nạ `m ≠ 0` luôn có chữ chung với nhau, nên không bao giờ là một cặp hợp lệ; và với mọi từ `x` khác, `x` rời với từ này ⇔ `x` rời với mọi từ cùng mặt nạ. Vậy trong các từ cùng mặt nạ chỉ cần giữ từ **dài nhất** (tích lớn nhất với mọi đối tác).
- **Cắt nhánh (Cách 4):** sắp theo độ dài giảm dần; tích `li · lj` giảm khi `j` tăng. Nếu `li · lj ≤ best` thì mọi `j' > j` cho tích `≤ best`: bỏ. Khi tìm được `j` rời nhau, các `j'` sau cho tích không lớn hơn nên `break` đúng. Điều kiện ở vòng ngoài: nếu `li · l_{i+1} ≤ best` thì mọi cặp `(i', j')` với `i' ≥ i` có tích `≤ li · l_{i+1}`.
- **Bitset (Cách 5):** `conflict` là các từ có chung ít nhất một chữ với từ hiện tại; `free` là phần bù; từ dài nhất trong `free` chọn bằng bit thấp nhất; xét mọi từ làm "chủ" nên bao gồm cặp tối ưu.
- **Không có cặp:** `best` giữ nguyên `0`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số từ, `L` tổng độ dài, `d` số mặt nạ khác nhau.

- **Cách 1:** Time `O(L + 26·n²)`; Space `O(26·n)`.
- **Cách 2:** Time `O(L + n²)`; Space `O(n)`.
- **Cách 3:** Time `O(L + d²)`; Space `O(d)`.
- **Cách 4:** Time `O(L + n log n)` thông thường, `O(n²)` xấu nhất; Space `O(n)`.
- **Cách 5:** Time `O(L + 26·n²/64)`; Space `O(n)`.
- Với `n = 1000`, `L ≤ 10⁶`: bước dựng mặt nạ `O(L)` chiếm phần lớn thời gian ở Cách 2.

## 12. Bộ test

Đối chiếu mọi cách với **brute force `set`** (độc lập với bitmask) trên từ ngẫu nhiên có seed, đủ loại bảng chữ cái (để có/không có cặp rời nhau).

In [14]:
impls = {
    "brute": max_product_brute,
    "masks": max_product_masks,
    "dedupe": max_product_dedupe,
    "pruned": max_product_pruned,
    "bitsets": max_product_bitsets,
    "submit": lambda words: Solution().maxProduct(words),
}


def check(words, expected=None):
    base = max_product_brute(words)
    if expected is not None:
        assert base == expected, (words, base, expected)
    for name, fn in impls.items():
        got = fn(list(words))
        assert got == base, (name, words, got, base)
    return base


for words, e in examples:
    check(words, e)
print("PASS 3 ví dụ của đề")

PASS 3 ví dụ của đề


### 12.1. Các trường hợp biên và giá trị tính tay

In [15]:
cases = [
    (["a", "b"], 1, "hai từ một chữ khác nhau"),
    (["a", "a"], 0, "hai từ giống nhau"),
    (["ab", "cd"], 4, "hai từ rời nhau"),
    (["ab", "ba", "cd"], 4, "hai từ cùng mặt nạ + một từ rời"),
    (["aaab", "ab", "cd"], 8, "mặt nạ trùng: phải giữ từ dài nhất (aaab, cd)"),
    (["abc", "bcd", "cde"], 0, "mọi cặp đều có chữ chung"),
    (["abc", "def", "ghi"], 9, "ba từ đôi một rời nhau"),
    (["aaaa", "bb"], 8, "độ dài tính cả chữ lặp"),
    (["ab", "bc"], 0, "chung chữ b"),
    (["z" * 1000, "a" * 1000], 10 ** 6, "độ dài tối đa: tích 10^6"),
    (["abcdefghijklmnopqrstuvwxyz", "a"], 0, "từ chứa cả 26 chữ"),
    (["abcdefghijklm", "nopqrstuvwxyz"], 169, "chia đôi bảng chữ cái 13 x 13"),
]
for words, e, why in cases:
    assert check(words, e) == e
    print("PASS", why)

PASS hai từ một chữ khác nhau
PASS hai từ giống nhau
PASS hai từ rời nhau
PASS hai từ cùng mặt nạ + một từ rời
PASS mặt nạ trùng: phải giữ từ dài nhất (aaab, cd)
PASS mọi cặp đều có chữ chung
PASS ba từ đôi một rời nhau
PASS độ dài tính cả chữ lặp
PASS chung chữ b
PASS độ dài tối đa: tích 10^6
PASS từ chứa cả 26 chữ
PASS chia đôi bảng chữ cái 13 x 13


### 12.2. Tính chất và phản ví dụ

In [16]:
rnd = random.Random(318)


def random_words(n, alphabet, maxlen, rnd):
    return ["".join(rnd.choice(alphabet) for _ in range(rnd.randint(1, maxlen))) for _ in range(n)]


# Tính chất 1: thứ tự các từ không ảnh hưởng
# Tính chất 2: thêm một từ không làm đáp án nhỏ đi (tính đơn điệu)
# Tính chất 3: đáp án là tích của hai độ dài có thật (hoặc 0); đổi tên chữ cái (hoán vị bảng chữ cái) không đổi đáp án
for _ in range(300):
    n = rnd.randint(2, 12)
    words = random_words(n, "abcdefgh", 6, rnd)
    base = max_product_masks(words)
    shuffled = words[:]
    rnd.shuffle(shuffled)
    assert max_product_masks(shuffled) == base
    extra = words + random_words(1, "abcdefgh", 6, rnd)
    assert max_product_masks(extra) >= base
    lens = {len(w) for w in words}
    assert base == 0 or any(base == a * b for a in lens for b in lens)
    perm = list("abcdefgh")
    rnd.shuffle(perm)
    table = str.maketrans("abcdefgh", "".join(perm))
    assert max_product_masks([w.translate(table) for w in words]) == base
print("PASS tính chất: bất biến theo thứ tự và đổi tên chữ cái; đơn điệu khi thêm từ; là tích hai độ dài")

# Phản ví dụ: 'số chữ cái khác nhau' không thay được 'độ dài'
words = ["aaaa", "bb"]
distinct_product = len(set(words[0])) * len(set(words[1]))
print("tích số chữ khác nhau =", distinct_product, "| tích độ dài đúng =", max_product_masks(words))
assert distinct_product != max_product_masks(words)

PASS tính chất: bất biến theo thứ tự và đổi tên chữ cái; đơn điệu khi thêm từ; là tích hai độ dài
tích số chữ khác nhau = 1 | tích độ dài đúng = 8


### 12.3. Đối chiếu ngẫu nhiên (có seed) với brute force

In [17]:
rnd = random.Random(2026)
for trial in range(1500):
    alphabet = rnd.choice(["ab", "abc", "abcd", "abcdef", "abcdefghijklmnopqrstuvwxyz"])
    n = rnd.randint(2, 15)
    check(random_words(n, alphabet, rnd.randint(1, 8), rnd))

for trial in range(30):                                   # n vừa
    alphabet = rnd.choice(["abcdefgh", "abcdefghijklm", "abcdefghijklmnopqrstuvwxyz"])
    check(random_words(rnd.randint(50, 200), alphabet, rnd.randint(1, 12), rnd))

for trial in range(3):                                    # lớn theo đề: n = 1000
    alphabet = rnd.choice(["abcdefgh", "abcdefghijklmnopqrstuvwxyz"])
    check(random_words(1000, alphabet, 30, rnd))

# 1000 từ rất dài (1000 ký tự): kiểm tra thời gian dựng mặt nạ
big = ["".join(rnd.choice("abcdefghijklm") for _ in range(1000)) for _ in range(500)]
big += ["".join(rnd.choice("nopqrstuvwxyz") for _ in range(1000)) for _ in range(500)]
assert check(big) == 10 ** 6
print("PASS 1500 test nhỏ, 30 test vừa, 3 test n = 1000, 1 test 1000 từ dài 1000 ký tự")

PASS 1500 test nhỏ, 30 test vừa, 3 test n = 1000, 1 test 1000 từ dài 1000 ký tự


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng số chữ cái khác nhau thay vì độ dài

In [18]:
def sai_so_chu_khac_nhau(words):
    masks = [word_mask(w) for w in words]
    best = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if masks[i] & masks[j] == 0:
                best = max(best, bin(masks[i]).count("1") * bin(masks[j]).count("1"))   # sai: phải len(words[i]) * len(words[j])
    return best


print("số chữ khác nhau ->", sai_so_chu_khac_nhau(["aaaa", "bb"]), "| đúng: 8")
assert sai_so_chu_khac_nhau(["aaaa", "bb"]) != 8

số chữ khác nhau -> 1 | đúng: 8


**Vì sao sai:** đề nhân **độ dài** của từ (tính cả chữ lặp), không phải số chữ cái khác nhau. `"aaaa"` dài 4.

### Lỗi 2: Dựng mặt nạ bằng `+=` thay vì `|=`

In [19]:
def sai_cong_mat_na(words):
    masks = []
    for w in words:
        m = 0
        for c in w:
            m += 1 << (ord(c) - 97)                       # sai: chữ lặp cộng nhiều lần, tràn sang bit khác
        masks.append(m)
    best = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if masks[i] & masks[j] == 0:
                best = max(best, len(words[i]) * len(words[j]))
    return best


print("dùng += ->", sai_cong_mat_na(["aa", "b"]), "| đúng: 2")
assert sai_cong_mat_na(["aa", "b"]) != 2

dùng += -> 0 | đúng: 2


**Vì sao sai:** `"aa"` có `m = 1 + 1 = 2`, trùng với mặt nạ của `"b"` (`2`): hai từ rời nhau bị coi là có chữ chung. Phải dùng `|=` (OR không cộng dồn).

### Lỗi 3: Đảo điều kiện `& == 0`

In [20]:
def sai_dao_dieu_kien(words):
    masks = [word_mask(w) for w in words]
    best = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if masks[i] & masks[j]:                       # sai: đây là CÓ chữ chung
                best = max(best, len(words[i]) * len(words[j]))
    return best


print("đảo điều kiện ->", sai_dao_dieu_kien(["ab", "bc"]), "| đúng: 0")
assert sai_dao_dieu_kien(["ab", "bc"]) != 0

đảo điều kiện -> 4 | đúng: 0


**Vì sao sai:** `mask_a & mask_b != 0` nghĩa là **có** chữ chung, trái với yêu cầu. Điều kiện đúng: `mask_a & mask_b == 0`. Lưu ý toán tử ưu tiên: viết `masks[i] & masks[j] == 0` hoạt động đúng trong Python (`&` ưu tiên cao hơn `==`), nhưng ở C/Java phải thêm ngoặc.

### Lỗi 4: Ghi đè mặt nạ trùng bằng độ dài ngắn hơn

In [21]:
def sai_ghi_de_mat_na(words):
    best_len = {}
    for w in words:
        best_len[word_mask(w)] = len(w)                   # sai: phải giữ độ dài LỚN nhất
    items = list(best_len.items())
    best = 0
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i][0] & items[j][0] == 0:
                best = max(best, items[i][1] * items[j][1])
    return best


print("ghi đè ->", sai_ghi_de_mat_na(["aaab", "ab", "cd"]), "| đúng: 8")
assert sai_ghi_de_mat_na(["aaab", "ab", "cd"]) != 8

ghi đè -> 4 | đúng: 8


**Vì sao sai:** `"aaab"` và `"ab"` cùng mặt nạ; bản sai giữ từ **xuất hiện sau** (`"ab"`, dài 2) thay vì từ dài nhất (`"aaab"`, dài 4) nên bỏ lỡ tích `4 · 2 = 8`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Hai chuỗi không có ký tự chung" với bảng chữ cái nhỏ → **bitmask**, `a & b == 0`.
- Dựng mặt nạ bằng `|=`, **không** `+=`.
- Cùng mặt nạ ⇒ chỉ cần giữ từ dài nhất (với bài "tối đa hóa độ dài").
- Cần giảm `O(n²)`: gộp mặt nạ, sắp + cắt nhánh, hoặc bitset theo chữ cái.
- Đọc kỹ: tích **độ dài** (có chữ lặp), không phải số chữ khác nhau.
- Kiểm thử: brute force `set` độc lập và chọn bảng chữ cái nhỏ để có nhiều cặp rời nhau lẫn va chạm.

## 15. Biến thể của bài

1. **Đếm số cặp** từ không có chữ chung.
2. **Tổng độ dài lớn nhất** của hai từ không có chữ chung.
3. **Trả về chỉ số** của cặp tối ưu.
4. **Cho phép tối đa `k` chữ chung** giữa hai từ.
5. **Hai từ có đúng cùng tập chữ cái** (anagram theo tập) — nhóm theo mặt nạ.

In [22]:
def count_disjoint_pairs(words):
    """Biến thể 1."""
    masks = [word_mask(w) for w in words]
    return sum(1 for i in range(len(words)) for j in range(i + 1, len(words)) if masks[i] & masks[j] == 0)


def max_sum_disjoint(words):
    """Biến thể 2: tổng độ dài lớn nhất (0 nếu không có cặp)."""
    masks = [word_mask(w) for w in words]
    best = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if masks[i] & masks[j] == 0:
                best = max(best, len(words[i]) + len(words[j]))
    return best


def best_pair_indices(words):
    """Biến thể 3: (i, j) của một cặp tối ưu (i < j), hoặc None."""
    masks = [word_mask(w) for w in words]
    best, arg = 0, None
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if masks[i] & masks[j] == 0 and len(words[i]) * len(words[j]) > best:
                best, arg = len(words[i]) * len(words[j]), (i, j)
    return arg


def max_product_at_most_k_common(words, k):
    """Biến thể 4: chữ chung (số chữ cái khác nhau chung) không quá k."""
    masks = [word_mask(w) for w in words]
    best = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if bin(masks[i] & masks[j]).count("1") <= k:
                best = max(best, len(words[i]) * len(words[j]))
    return best


def group_by_mask(words):
    """Biến thể 5: nhóm các từ có cùng tập chữ cái."""
    groups = {}
    for w in words:
        groups.setdefault(word_mask(w), []).append(w)
    return sorted(sorted(g) for g in groups.values())


rnd = random.Random(15)
for _ in range(300):
    words = random_words(rnd.randint(2, 10), "abcdef", 5, rnd)
    sets = [set(w) for w in words]
    pairs = [(i, j) for i in range(len(words)) for j in range(i + 1, len(words))]
    assert count_disjoint_pairs(words) == sum(sets[i].isdisjoint(sets[j]) for i, j in pairs)
    assert max_sum_disjoint(words) == max([len(words[i]) + len(words[j]) for i, j in pairs if sets[i].isdisjoint(sets[j])], default=0)
    arg = best_pair_indices(words)
    if arg is None:
        assert max_product_masks(words) == 0
    else:
        assert len(words[arg[0]]) * len(words[arg[1]]) == max_product_masks(words) and sets[arg[0]].isdisjoint(sets[arg[1]])
    assert max_product_at_most_k_common(words, 0) == max_product_masks(words)
    assert max_product_at_most_k_common(words, 0) <= max_product_at_most_k_common(words, 2)
    assert max_product_at_most_k_common(words, 6) == max(len(words[i]) * len(words[j]) for i, j in pairs)
assert group_by_mask(["ab", "ba", "aab", "c"]) == [["aab", "ab", "ba"], ["c"]]
print("PASS biến thể: đếm cặp, tổng độ dài, chỉ số cặp, tối đa k chữ chung, nhóm theo mặt nạ")

PASS biến thể: đếm cặp, tổng độ dài, chỉ số cặp, tối đa k chữ chung, nhóm theo mặt nạ


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cặp + `set` | `isdisjoint` mỗi cặp | `O(L + 26·n²)` | `O(26·n)` | oracle |
| 2. Cặp + bitmask | `mask_i & mask_j == 0` | `O(L + n²)` | `O(n)` | **cách nộp bài** |
| 3. Gộp mặt nạ | giữ độ dài lớn nhất mỗi mặt nạ | `O(L + d²)` | `O(d)` | nhiều từ trùng mặt nạ |
| 4. Sắp + cắt nhánh | độ dài giảm dần, dừng sớm | `O(L + n log n)` thường | `O(n)` | dữ liệu "dễ" |
| 5. Bitset theo chữ cái | `free = ~OR(bits[c])`, bit thấp nhất | `O(L + 26·n²/64)` | `O(n)` | `n` lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `word_mask("ab")`, `word_mask("ab") & word_mask("cd")` và `max_product_masks(["ab", "cd"])`.

**Bài 2 (tính tay).** Với `["a","ab","abc","d","cd","bcd","abcd"]`, liệt kê mọi cặp rời nhau và tích của chúng.

**Bài 3 (điền TODO).** Hoàn thành `max_product_todo` theo Cách 2.

In [23]:
def max_product_todo(words):
    # TODO 1: masks = []; với mỗi từ dựng mask bằng |= (1 << (ord(c) - 97))
    # TODO 2: lens = [len(w) for w in words]; best = 0
    # TODO 3: với mọi cặp i < j: nếu masks[i] & masks[j] == 0 thì best = max(best, lens[i] * lens[j])
    # TODO 4: trả best
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_product_dedupe2(words)` theo Cách 3 (dict `mặt nạ → độ dài lớn nhất`, sau đó duyệt cặp các mặt nạ khác nhau).

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_disjoint_pairs2(words)` (đếm cặp rời nhau). (b) Viết `max_sum_disjoint2(words)` (tổng độ dài lớn nhất).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `a` là bit 0, `b` là bit 1; hai mặt nạ rời nhau thì AND bằng 0. Bài 2: kiểm từng cặp xem có chữ chung không.
- **Mức 2 (vừa):** Bài 1: `3`, `0`, `4`. Bài 2: `a-d` 1; `a-cd` 2; `a-bcd` 3; `ab-d` 2; `ab-cd` 4; `abc-d` 3; lớn nhất `4`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: dùng `if len(w) > best_len.get(m, 0): best_len[m] = len(w)`. Bài 5: giống `max_product_masks` nhưng đổi phép cập nhật (đếm / cộng).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert word_mask("ab") == 3
assert word_mask("ab") & word_mask("cd") == 0
assert max_product_masks(["ab", "cd"]) == 4

# Bài 2
words = ["a", "ab", "abc", "d", "cd", "bcd", "abcd"]
pairs = [(words[i], words[j], len(words[i]) * len(words[j]))
         for i in range(len(words)) for j in range(i + 1, len(words))
         if set(words[i]).isdisjoint(words[j])]
assert sorted(p[2] for p in pairs) == [1, 2, 2, 3, 3, 4]
assert max(p[2] for p in pairs) == 4

# Bài 3
def max_product_todo_ans(words):
    masks = []
    for w in words:
        m = 0
        for c in w:
            m |= 1 << (ord(c) - 97)
        masks.append(m)
    lens = [len(w) for w in words]
    best = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if masks[i] & masks[j] == 0:
                best = max(best, lens[i] * lens[j])
    return best


# Bài 4
def max_product_dedupe2(words):
    best_len = {}
    for w in words:
        m = word_mask(w)
        if len(w) > best_len.get(m, 0):
            best_len[m] = len(w)
    items = list(best_len.items())
    return max((li * lj for i, (mi, li) in enumerate(items) for mj, lj in items[i + 1:] if mi & mj == 0), default=0)


# Bài 5
def count_disjoint_pairs2(words):
    masks = [word_mask(w) for w in words]
    return sum((masks[i] & masks[j]) == 0 for i in range(len(words)) for j in range(i + 1, len(words)))


def max_sum_disjoint2(words):
    masks = [word_mask(w) for w in words]
    return max((len(words[i]) + len(words[j]) for i in range(len(words)) for j in range(i + 1, len(words))
                if masks[i] & masks[j] == 0), default=0)


rnd = random.Random(19)
for _ in range(300):
    words = random_words(rnd.randint(2, 12), "abcdefg", 6, rnd)
    base = max_product_brute(words)
    assert max_product_todo_ans(words) == base
    assert max_product_dedupe2(words) == base
    assert count_disjoint_pairs2(words) == count_disjoint_pairs(words)
    assert max_sum_disjoint2(words) == max_sum_disjoint(words)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (bitmask cho mọi cặp)

```python
class Solution:
    def maxProduct(self, words: List[str]) -> int:
        n = len(words)
        masks = [0] * n
        lens = [len(w) for w in words]
        for i, w in enumerate(words):
            for c in w:
                masks[i] |= 1 << (ord(c) - ord("a"))
        best = 0
        for i in range(n):
            for j in range(i + 1, n):
                if masks[i] & masks[j] == 0:
                    best = max(best, lens[i] * lens[j])
        return best
```

**Checklist**

- [ ] Mặt nạ dựng bằng `|=` (không phải `+=`)?
- [ ] Điều kiện rời nhau là `mask_i & mask_j == 0`?
- [ ] Tích dùng **độ dài** từ (`len`), không phải số chữ cái khác nhau?
- [ ] Không có cặp nào ⇒ trả `0`?
- [ ] Nếu gộp mặt nạ: giữ độ dài **lớn nhất**, không ghi đè bừa?
- [ ] Đã thử: hai từ giống nhau, từ chứa 26 chữ, từ dài 1000 ký tự, mặt nạ trùng?
- [ ] Đã đối chiếu với brute force `set` trên input ngẫu nhiên có seed?
- [ ] Skeleton đúng `class Solution` / `maxProduct(self, words: List[str]) -> int`?